# RTL-to-GDS full-chip with a custom macro on GF180MCU

Walk-through of the **full-chip (padring) flow** on the wafer-space
`gf180mcu-project-template`, with one of the two on-die SRAMs
replaced by a **custom 4-bit counter macro** that was hardened
separately in the Classic flow. Builds on the sibling notebook
`rtl2gds_counter.ipynb` (bare-block Classic flow, same repo).

**Pedagogical goal.** Show the hierarchical pattern students will
use in real projects:
1. **Harden each block** (counter, ALU, …) on its own with the
   Classic flow. Each produces `.gds + .lef + .lib + .v blackbox`.
2. **Drop the blocks** into the full-chip `config.yaml` as
   pre-characterised `MACROS`, alongside foundry macros (SRAMs) and
   wafer-space IP (chip-ID, logo).
3. **Instantiate them** in the chip top-level RTL.
4. **Run the full-chip flow** once; LibreLane handles placement,
   routing, and signoff over the whole tile.

> **Known limitation — chip-top LVS.** This walk-through goes
> Magic-DRC clean (0 errors) but Netgen LVS at chip-top fails the
> port-matching check on the wafer-space `slot_1x1` template because
> Magic's GDS extraction emits the top cell with **only `VSS` as a
> top-level port** (the Verilog netlist has both `VDD` and `VSS`).
> The IR-drop step confirms `VDD` is electrically present at
> ~5 V everywhere; the issue is at the boundary-label / port
> recognition layer of the wafer-space `slot_1x1` floorplan, not in
> the macro replacement that this example teaches. The same
> pedagogical recipe (harden a block, drop it under `MACROS:`, patch
> `PDN_MACRO_CONNECTIONS` + `pdn_cfg.tcl`) gets you to **fully clean
> signoff** in `examples/04_counter_alu_multimacro/` (different fork:
> the chipathon-2026 padring), so for tape-out follow #04. The macro
> mechanics shown here are correct as-is.

**Not self-contained.** Unlike the counter notebook, this one
needs an on-disk clone of the wafer-space template (several
hundred MB) and the wafer-space GF180MCU PDK fork. The notebook
clones them into `~/eda/designs/chip_custom/` if you flip
`RUN_CLONE_TEMPLATE = True`, otherwise it assumes they are there.

**Runtime for the flow itself:** ~80 min end-to-end on a modern
host with KLayout DRC disabled (Step 4 patches it because gf180mcuD
ships no curated KLayout DRC runset and Magic DRC is authoritative;
Magic DRC alone takes ~30 min on this design). Every heavy step has
a `RUN_*` flag defaulted to `False` so you can rehearse with a dry
run first.

**Prerequisites.** Linux host, Docker, ~40 GB free disk, the
counter artifacts already on disk from the sibling notebook
(`~/eda/designs/counter_demo/runs/demo/final/`). If you do not
have those, run `rtl2gds_counter.ipynb` first (1–2 min).

## Step 0 — configuration

All paths, names, and `RUN_*` flags live here. Defaults are safe:
nothing destructive runs until you flip a flag.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import textwrap

# ---- flags: flip to True to actually execute a step ----
RUN_CLONE_TEMPLATE  = False   # ~250 MB, ~30 s
RUN_CLONE_PDK_FORK  = False   # ~500 MB, ~1-2 min
RUN_COPY_COUNTER    = False   # cheap (six file copies)
RUN_PATCH_SOURCES   = False   # text patches on 3 files
RUN_FLOW            = False   # ~35-45 min, 18-20 GB runs dir

# ---- container (already running) ----
CONTAINER_NAME = "gf180"

# ---- host paths ----
HOST_WORKSPACE    = Path.home() / "eda" / "designs"
HOST_PROJECT_DIR  = HOST_WORKSPACE / "chip_custom"
HOST_TEMPLATE     = HOST_PROJECT_DIR / "template"
HOST_COUNTER_MACRO = HOST_TEMPLATE / "counter_macro"
HOST_COUNTER_SRC  = HOST_WORKSPACE / "counter_demo" / "runs" / "demo" / "final"

# ---- container paths (bind-mount is ~/eda/designs ↔ /foss/designs) ----
CONTAINER_PROJECT  = "/foss/designs/chip_custom"
CONTAINER_TEMPLATE = f"{CONTAINER_PROJECT}/template"

# ---- GF180MCU identifiers ----
PDK_NAME     = "gf180mcuD"
STD_CELL_LIB = "gf180mcu_fd_sc_mcu7t5v0"

# ---- wafer-space upstream references ----
TEMPLATE_REPO = "https://github.com/wafer-space/gf180mcu-project-template.git"
PDK_FORK_REPO = "https://github.com/wafer-space/gf180mcu.git"
PDK_FORK_TAG  = "1.8.0"


def run_or_print(cmd, do_it, *, shell_on_container=False, timeout=None, cwd=None):
    """Print the command; execute only if do_it is True."""
    if shell_on_container:
        print(f"$ docker exec {CONTAINER_NAME} bash -lc '<script>'")
        print(textwrap.indent(cmd, "  | "))
    else:
        print("$ " + " ".join(cmd) + (f"   (cwd={cwd})" if cwd else ""))
    if not do_it:
        print("  (skipped -- flip the RUN_* flag to execute)\n")
        return None
    print("  (executing...)")
    args = (
        ["docker", "exec", CONTAINER_NAME, "bash", "-lc", cmd]
        if shell_on_container else cmd
    )
    proc = subprocess.run(args, capture_output=True, text=True, timeout=timeout, cwd=cwd)
    if proc.stdout.strip():
        print(proc.stdout[-4000:])
    if proc.returncode != 0 and proc.stderr.strip():
        print("STDERR (tail):")
        print(proc.stderr[-2000:])
    print(f"  returncode={proc.returncode}\n")
    return proc


HOST_PROJECT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Workspace on host: {HOST_WORKSPACE}")
print(f"Project dir:       {HOST_PROJECT_DIR}")
print(f"Template:          {HOST_TEMPLATE}")
print(f"Counter macro:     {HOST_COUNTER_MACRO}")
print(f"Counter source:    {HOST_COUNTER_SRC}")
print(f"PDK (inside ctr):  /foss/pdks/{PDK_NAME}  (built into image)")
print(f"Container project: {CONTAINER_PROJECT}")

## Step 1 — prerequisites

Before anything else we check:

1. The IIC-OSIC-TOOLS Docker image is pulled and the `gf180`
   container is running with the workspace bind-mount.
2. The counter hardening artifacts exist (from the sibling
   `rtl2gds_counter.ipynb`).

The cell below is read-only — it just prints what it found.

In [ ]:
def check(label, ok, detail=""):
    tag = "OK " if ok else "!! "
    print(f"{tag}{label}" + (f"  -- {detail}" if detail else ""))
    return ok

all_ok = True

# Docker image
proc = subprocess.run(
    ["docker", "image", "inspect", "hpretl/iic-osic-tools:next"],
    capture_output=True, text=True,
)
all_ok &= check("Docker image hpretl/iic-osic-tools:next present", proc.returncode == 0)

# Container running
proc = subprocess.run(
    ["docker", "ps", "--filter", f"name={CONTAINER_NAME}", "--format", "{{.Names}}"],
    capture_output=True, text=True,
)
all_ok &= check(f"Container '{CONTAINER_NAME}' running", CONTAINER_NAME in proc.stdout)

# Counter artifacts
needed = [
    HOST_COUNTER_SRC / "gds"  / "counter.gds",
    HOST_COUNTER_SRC / "lef"  / "counter.lef",
    HOST_COUNTER_SRC / "lib"  / "nom_tt_025C_5v00" / "counter__nom_tt_025C_5v00.lib",
    HOST_COUNTER_SRC / "lib"  / "nom_ss_125C_4v50" / "counter__nom_ss_125C_4v50.lib",
    HOST_COUNTER_SRC / "lib"  / "nom_ff_n40C_5v50" / "counter__nom_ff_n40C_5v50.lib",
]
for p in needed:
    all_ok &= check(f"{p.name}", p.exists(), str(p))

if not all_ok:
    print("\nFix the missing items above before running the flow.")
    print("Tip: run rtl2gds_counter.ipynb first if counter artifacts are absent.")

## Step 2 — clone the template, clone the PDK fork, copy counter artifacts

Three things happen here:

1. **Clone** `github.com/wafer-space/gf180mcu-project-template.git`
   into `~/eda/designs/chip_custom/template/` (shallow). Contains
   the full-chip skeleton: `src/chip_top.sv`, `src/chip_core.sv`,
   `librelane/config.yaml`, `librelane/slots/*.yaml`, `Makefile`,
   plus wafer-space IP (chip-ID, logo) and the cocotb testbench.
2. **Clone** the wafer-space GF180MCU PDK fork at tag `1.8.0`. This
   adds the two padring I/O cells (`gf180mcu_ws_io__dvdd/dvss`)
   that upstream `gf180mcu-pdk` does not ship. Lives inside the
   template at `template/gf180mcu/`.
3. **Copy** the counter hardening artifacts into
   `template/counter_macro/` plus a minimal blackbox `counter.v`
   for synthesis.

Leave all three `RUN_*` flags `False` on a dry run.

In [ ]:
# 2a -- clone template
if HOST_TEMPLATE.exists():
    print(f"Template already at {HOST_TEMPLATE}  (skipping clone)")
else:
    run_or_print(
        ["git", "clone", "--depth", "1", TEMPLATE_REPO, str(HOST_TEMPLATE)],
        RUN_CLONE_TEMPLATE,
    )

# 2b -- clone wafer-space PDK fork inside the template
pdk_fork_dir = HOST_TEMPLATE / "gf180mcu"
if pdk_fork_dir.exists():
    print(f"Wafer-space PDK fork already at {pdk_fork_dir}  (skipping clone)")
else:
    run_or_print(
        ["git", "clone", "--depth", "1", "--branch", PDK_FORK_TAG,
         PDK_FORK_REPO, str(pdk_fork_dir)],
        RUN_CLONE_PDK_FORK,
    )

# 2c -- copy counter artifacts + write blackbox
copies = [
    (HOST_COUNTER_SRC / "gds"  / "counter.gds",                           "counter.gds"),
    (HOST_COUNTER_SRC / "lef"  / "counter.lef",                           "counter.lef"),
    (HOST_COUNTER_SRC / "lib"  / "nom_tt_025C_5v00" / "counter__nom_tt_025C_5v00.lib", "counter__nom_tt_025C_5v00.lib"),
    (HOST_COUNTER_SRC / "lib"  / "nom_ss_125C_4v50" / "counter__nom_ss_125C_4v50.lib", "counter__nom_ss_125C_4v50.lib"),
    (HOST_COUNTER_SRC / "lib"  / "nom_ff_n40C_5v50" / "counter__nom_ff_n40C_5v50.lib", "counter__nom_ff_n40C_5v50.lib"),
]
COUNTER_BLACKBOX = """\
module counter (
    input  wire       clk,
    input  wire       rst,
    output wire [3:0] q
);
endmodule
"""

if RUN_COPY_COUNTER:
    HOST_COUNTER_MACRO.mkdir(parents=True, exist_ok=True)
    for src, dst_name in copies:
        shutil.copy2(src, HOST_COUNTER_MACRO / dst_name)
    (HOST_COUNTER_MACRO / "counter.v").write_text(COUNTER_BLACKBOX)
    print("Copied:")
    for f in HOST_COUNTER_MACRO.iterdir():
        print(f"  {f.name:40s}  {f.stat().st_size} B")
else:
    print("Would copy (flip RUN_COPY_COUNTER = True):")
    for src, dst_name in copies:
        print(f"  {src} -> {HOST_COUNTER_MACRO / dst_name}")
    print(f"Would write blackbox at {HOST_COUNTER_MACRO / 'counter.v'} :")
    print(textwrap.indent(COUNTER_BLACKBOX, "    "))

## Step 3 — patch `chip_core.sv` (replace `sram_0` with the counter)

The template's `src/chip_core.sv` instantiates two SRAMs in
scratch mode (`CEN=1`, outputs XORed into the bidir pads). We
leave `sram_1` untouched and replace `sram_0` with our counter
macro. To keep the downstream XOR width intact, we reuse the name
`sram_0_out` as an 8-bit wire that zero-extends the counter's
4-bit output.

The patch is a plain string substitution; if you re-run this cell
after a successful patch the second attempt is a no-op.

In [ ]:
chip_core_path = HOST_TEMPLATE / "src" / "chip_core.sv"

ORIGINAL_SRAM0 = """\
    logic [7:0] sram_0_out;

    gf180mcu_fd_ip_sram__sram512x8m8wm1 sram_0 (
        `ifdef USE_POWER_PINS
        .VDD  (VDD),
        .VSS  (VSS),
        `endif

        .CLK  (clk),
        .CEN  (1'b1),
        .GWEN (1'b0),
        .WEN  (8'b0),
        .A    ('0),
        .D    ('0),
        .Q    (sram_0_out)
    );"""

PATCHED_COUNTER = """\
    // Custom macro instance: 4-bit counter (replaces sram_0 for this demo).
    wire [3:0] counter_out;

    counter counter_0 (
        .clk(clk),
        .rst(~rst_n),
        .q  (counter_out)
    );

    // Zero-extend counter_out from 4 bits to 8 to keep the XOR width below.
    wire [7:0] sram_0_out = {4'd0, counter_out};"""

if not chip_core_path.exists():
    print(f"{chip_core_path} not found. Clone the template first (Step 2a).")
else:
    src = chip_core_path.read_text()
    already = PATCHED_COUNTER.splitlines()[0].strip() in src
    in_orig  = ORIGINAL_SRAM0.splitlines()[2].strip() in src
    print(f"Patch already applied: {already}")
    print(f"Original sram_0 present: {in_orig}\n")
    print("Replacement block:")
    print(textwrap.indent(PATCHED_COUNTER, "    "))
    if RUN_PATCH_SOURCES and in_orig and not already:
        new_src = src.replace(ORIGINAL_SRAM0, PATCHED_COUNTER)
        chip_core_path.write_text(new_src)
        print(f"\n-> Patched {chip_core_path}")
    elif RUN_PATCH_SOURCES:
        print("\n(No change needed.)")
    else:
        print("\n(Dry run; flip RUN_PATCH_SOURCES = True to apply.)")

## Step 4 — patch `config.yaml` and `pdn_cfg.tcl`

Three surgical patches:

1. In `librelane/config.yaml`, the `MACROS:` block gains a new
   `counter` entry pointing at `../counter_macro/`, and the
   SRAM entry drops `i_chip_core.sram_0` from its `instances:`
   list (only `sram_1` remains). `PDN_MACRO_CONNECTIONS` is updated
   from `i_chip_core.sram_0` to `i_chip_core.counter_0`.
2. In `librelane/pdn_cfg.tcl`, the per-macro PDN grid that was
   attached to `i_chip_core.sram_0` is re-pointed at
   `i_chip_core.counter_0`. The extra Metal4 stripes are left as
   is — they are oversized for a 300x300 um counter but harmless.
3. In `librelane/config.yaml` `meta.substituting_steps`, uncomment
   `KLayout.DRC: null` + `Checker.KLayoutDRC: null`. The wafer-space
   template ships these commented because gf180mcuD has no curated
   KLayout DRC runset; running open-source KLayout DRC on a chip-top
   takes 1+ hours (loading 11M+ via polygons single-threaded) and
   adds no value because Magic DRC is the authoritative DRC for this
   PDK. The same idea is applied across `examples/01-04`.

In [ ]:
config_path = HOST_TEMPLATE / "librelane" / "config.yaml"
pdn_path    = HOST_TEMPLATE / "librelane" / "pdn_cfg.tcl"

# ---- config.yaml patch ----
CONFIG_OLD = """\
  gf180mcu_fd_ip_sram__sram512x8m8wm1:
    gds:
      - pdk_dir::libs.ref/gf180mcu_fd_ip_sram/gds/gf180mcu_fd_ip_sram__sram512x8m8wm1.gds
    lef:
      - pdk_dir::libs.ref/gf180mcu_fd_ip_sram/lef/gf180mcu_fd_ip_sram__sram512x8m8wm1.lef
    vh:
      - pdk_dir::libs.ref/gf180mcu_fd_ip_sram/verilog/gf180mcu_fd_ip_sram__sram512x8m8wm1__blackbox.v
    lib:
      \"*_tt_025C_5v00\":
        - pdk_dir::libs.ref/gf180mcu_fd_ip_sram/lib/gf180mcu_fd_ip_sram__sram512x8m8wm1__tt_025C_5v00.lib
      \"*_ff_n40C_5v50\":
        - pdk_dir::libs.ref/gf180mcu_fd_ip_sram/lib/gf180mcu_fd_ip_sram__sram512x8m8wm1__ff_n40C_5v50.lib
      \"*_ss_125C_4v50\":
        - pdk_dir::libs.ref/gf180mcu_fd_ip_sram/lib/gf180mcu_fd_ip_sram__sram512x8m8wm1__ss_125C_4v50.lib
    instances:
      # Make sure to specify the SRAMs in pdn_cfg.tcl
      i_chip_core.sram_0:
        location: [490, 500]
        orientation: N
      i_chip_core.sram_1:
        location: [970, 500]
        orientation: E

# This is an alternative method compared to
# specifying the power connections in the RTL
PDN_MACRO_CONNECTIONS:
- \"i_chip_core.sram_0 VDD VSS VDD VSS\"
- \"i_chip_core.sram_1 VDD VSS VDD VSS\""""

CONFIG_NEW = """\
  # Custom counter macro (replaces sram_0) -- hardened separately in the
  # Classic flow and dropped in here as a pre-characterised block.
  counter:
    gds:
      - dir::../counter_macro/counter.gds
    lef:
      - dir::../counter_macro/counter.lef
    vh:
      - dir::../counter_macro/counter.v
    lib:
      \"*_tt_025C_5v00\":
        - dir::../counter_macro/counter__nom_tt_025C_5v00.lib
      \"*_ff_n40C_5v50\":
        - dir::../counter_macro/counter__nom_ff_n40C_5v50.lib
      \"*_ss_125C_4v50\":
        - dir::../counter_macro/counter__nom_ss_125C_4v50.lib
    instances:
      i_chip_core.counter_0:
        location: [490, 500]
        orientation: N

  gf180mcu_fd_ip_sram__sram512x8m8wm1:
    gds:
      - pdk_dir::libs.ref/gf180mcu_fd_ip_sram/gds/gf180mcu_fd_ip_sram__sram512x8m8wm1.gds
    lef:
      - pdk_dir::libs.ref/gf180mcu_fd_ip_sram/lef/gf180mcu_fd_ip_sram__sram512x8m8wm1.lef
    vh:
      - pdk_dir::libs.ref/gf180mcu_fd_ip_sram/verilog/gf180mcu_fd_ip_sram__sram512x8m8wm1__blackbox.v
    lib:
      \"*_tt_025C_5v00\":
        - pdk_dir::libs.ref/gf180mcu_fd_ip_sram/lib/gf180mcu_fd_ip_sram__sram512x8m8wm1__tt_025C_5v00.lib
      \"*_ff_n40C_5v50\":
        - pdk_dir::libs.ref/gf180mcu_fd_ip_sram/lib/gf180mcu_fd_ip_sram__sram512x8m8wm1__ff_n40C_5v50.lib
      \"*_ss_125C_4v50\":
        - pdk_dir::libs.ref/gf180mcu_fd_ip_sram/lib/gf180mcu_fd_ip_sram__sram512x8m8wm1__ss_125C_4v50.lib
    instances:
      i_chip_core.sram_1:
        location: [970, 500]
        orientation: E

# This is an alternative method compared to
# specifying the power connections in the RTL
PDN_MACRO_CONNECTIONS:
- \"i_chip_core.counter_0 VDD VSS VDD VSS\"
- \"i_chip_core.sram_1 VDD VSS VDD VSS\""""

# Un-escape the \" sequences so the strings match file contents exactly.
CONFIG_OLD = CONFIG_OLD.replace('\\"', '"')
CONFIG_NEW = CONFIG_NEW.replace('\\"', '"')

# ---- KLayout DRC disable patch ----
# gf180mcuD ships no curated KLayout runset and the open-source version
# can take 1+ hours on chip-top designs (loading 11M+ via polygons).
# Magic DRC is the authoritative DRC for this PDK. The wafer-space
# template ships these lines already commented-out under
# meta.substituting_steps; we uncomment them here.
KLAYOUT_DRC_OLD = """\
    # Disable KLayout DRC
    #KLayout.DRC: null
    #Checker.KLayoutDRC: null"""

KLAYOUT_DRC_NEW = """\
    # Disable KLayout DRC
    # gf180mcuD ships no curated KLayout runset and the open-source version
    # can take 1+ hours on chip-top designs. Magic DRC is authoritative.
    KLayout.DRC: null
    Checker.KLayoutDRC: null"""

# ---- pdn_cfg.tcl patch ----
PDN_OLD = "    -instances i_chip_core.sram_0 \\\n"
PDN_NEW = "    -instances i_chip_core.counter_0 \\\n"

# Apply
for path, old, new in [(config_path, CONFIG_OLD,       CONFIG_NEW),
                        (config_path, KLAYOUT_DRC_OLD,  KLAYOUT_DRC_NEW),
                        (pdn_path,    PDN_OLD,          PDN_NEW)]:
    if not path.exists():
        print(f"{path} not found -- skipping.")
        continue
    src = path.read_text()
    already = new.splitlines()[0].strip() in src and new.splitlines()[-1].strip() in src
    in_orig  = old.splitlines()[0].strip() in src if old.strip() else False
    print(f"{path.name:20s}  already-patched={already}  original-present={in_orig}")
    if RUN_PATCH_SOURCES and in_orig and not already:
        path.write_text(src.replace(old, new))
        print(f"  -> patched")

if not RUN_PATCH_SOURCES:
    print("\n(Dry run; flip RUN_PATCH_SOURCES = True to apply.)")

## Step 5 — run the full-chip flow

Three actions inside the container:

1. `cd` into the template directory (lives in the bind-mount).
2. `source sak-pdk-script.sh gf180mcuD gf180mcu_fd_sc_mcu7t5v0` to
   switch the container's active PDK to GF180 (the image defaults
   to IHP-SG13G2).
3. `make librelane SLOT=1x1 PDK=gf180mcuD PDK_ROOT=.../gf180mcu`.
   The Makefile wraps `librelane slots/slot_1x1.yaml config.yaml
   --pdk gf180mcuD --pdk-root <wafer-space fork> --manual-pdk`.

Expected wall time: 35–45 min. The flow writes run artifacts into
`librelane/runs/RUN_<timestamp>/` and copies signoff views into
`final/`.

In [ ]:
flow_script = textwrap.dedent(f"""
    set -e
    cd {CONTAINER_TEMPLATE}
    source sak-pdk-script.sh {PDK_NAME} {STD_CELL_LIB}
    make librelane \\
        SLOT=1x1 \\
        PDK={PDK_NAME} \\
        PDK_ROOT={CONTAINER_TEMPLATE}/gf180mcu
""").strip()

# No timeout: chip-top flow can run multiple hours (Magic DRC alone
# is ~30-60 min on this PDK). Interrupt the cell with the kernel's
# stop button if you need to abort. To monitor progress from another
# shell:
#   docker exec gf180 bash -lc 'ls /foss/designs/chip_custom/template/librelane/runs/ | tail -1 | xargs -I{{}} ls /foss/designs/chip_custom/template/librelane/runs/{{}} | grep -E "^[0-9]+-" | tail'
run_or_print(flow_script, RUN_FLOW, shell_on_container=True, timeout=None)

## Step 6 — show the new full-chip render

LibreLane's `KLayout.Render` step writes `final/render/chip_top.png`
at the end of a successful run. We display it inline. For an
interactive view open the GDS on the host:

```bash
klayout ~/eda/designs/chip_custom/template/final/gds/chip_top.gds
```

In [ ]:
from IPython.display import Image, display

png_path = HOST_TEMPLATE / "final" / "render" / "chip_top.png"
gds_path = HOST_TEMPLATE / "final" / "gds"    / "chip_top.gds"

if png_path.exists():
    print(f"LibreLane render: {png_path}")
    display(Image(str(png_path)))
else:
    print(f"Render PNG not found: {png_path}")
    if gds_path.exists():
        print(f"GDS is there though: {gds_path}")
        print(f"  open it on the host with:  klayout {gds_path}")
    else:
        print("No GDS either -- Step 5 has not produced artifacts yet.")

## Step 7 — compare metrics: SRAM-only vs counter + SRAM

Key metrics to contrast between the unmodified template run and
ours: standard-cell instance count, core utilisation, macro count,
antenna violations, setup/hold across all corners, total power.

If you also have the reference run's `metrics.csv` from the
unmodified template (e.g. at
`/tmp/gf180-chip-test/template/final/metrics.csv`), the cell below
prints a side-by-side. Otherwise it just dumps our metrics.

In [ ]:
import csv

our_metrics = HOST_TEMPLATE / "final" / "metrics.csv"
ref_metrics = Path("/tmp/gf180-chip-test/template/final/metrics.csv")

keys = [
    "design__instance__count",
    "design__instance__count__stdcell",
    "design__instance__count__class:macro",
    "design__core_util",
    "magic__drc_error__count",
    "klayout__drc_error__count",
    "design__lvs_error__count",
    "antenna__violating__nets",
    "timing__setup_vio__count",
    "timing__hold_vio__count",
    "power__total",
    "route__wirelength__estimated",
]

def read_metrics(path):
    m = {}
    if not path.exists():
        return m
    with path.open() as fh:
        for row in csv.reader(fh):
            if row and row[0] in keys:
                m[row[0]] = row[1] if len(row) > 1 else ""
    return m

ours = read_metrics(our_metrics)
ref  = read_metrics(ref_metrics)

if not ours:
    print(f"Our metrics.csv not found: {our_metrics}")
    print("Run the flow first (Step 5).")
else:
    print(f"{'metric':45s}  {'reference (SRAM x2)':22s}  {'ours (counter + SRAM)':22s}")
    print("-" * 92)
    for k in keys:
        r = ref.get(k, "(n/a)") if ref else "(no ref)"
        o = ours.get(k, "(missing)")
        print(f"{k:45s}  {r:22s}  {o:22s}")

## Cleanup and next steps

Everything the flow produced lives under
`~/eda/designs/chip_custom/template/`. The `librelane/runs/`
directory is huge (~20 GB). Good practice:

```bash
# Keep only the final/ signoff artifacts; drop the intermediate run.
rm -rf ~/eda/designs/chip_custom/template/librelane/runs

# Stop the container if you are done with Docker (host files survive).
docker stop gf180
```

### Try next (stepping up to scope C)

1. **Add a second custom macro (ALU)**. Harden an ALU with
   `rtl2gds_counter.ipynb` as a template (swap the counter RTL for
   an ALU design). Then:
   - Add a second entry to the `MACROS:` block.
   - Replace the remaining `sram_1` in `chip_core.sv`.
   - Update `PDN_MACRO_CONNECTIONS` and `pdn_cfg.tcl`.
2. **Wire the counter output to the bidir pads directly**
   (remove the XOR with `count`). Tiny RTL change, no floorplan
   change — the counter value appears on bidirectional output pins.
3. **Tighten `CLOCK_PERIOD`** in the counter hardening and re-run
   both flows. Watch setup/hold slack move at the full-chip
   signoff.